# Week 5 — Flask API + Deployment Integration Tests

**Goal**: Verify the full stack works end-to-end before pushing to HuggingFace Spaces.

This notebook is shorter than previous weeks by design — most of the Week 5 work is in terminal commands and file edits, not notebooks. Use this to:
1. Run integration tests against the live Flask server
2. Profile inference performance 
3. Check the Streamlit app renders correctly
4. Confirm all deployment files are in place

**Run `pytest tests/test_predict.py -v` first — all 45 tests must pass before you deploy.**

## 0. Imports + config

In [ ]:
import sys, os, json, time, subprocess
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

from app.predict import load_models, predict_text, RISK_META

plt.rcParams.update({'figure.dpi':130,'font.family':'DejaVu Sans'})
DOCS = '../docs'
print('Setup complete.')

## 1. Load models + smoke test

In [ ]:
models = load_models()
print(f'Classes  : {models["le"].classes_}')
print(f'Vocab    : {len(models["vec"].vocabulary_)} tokens')
print(f'Loaded at: {models["loaded_at"]}')

# Quick smoke test
test_texts = [
    ('high',     'I feel completely hopeless and cannot see any reason to continue.'),
    ('low',      'Had a great productive week, feeling energised and grateful for support.'),
    ('moderate', 'Struggling a bit this week, not sure how to handle everything piling up.'),
]
print('\nSmoke test:')
for expected, text in test_texts:
    r = predict_text(text, models)
    status = '✅' if r['label'] == expected else '⚠'
    print(f'  {status} Expected={expected:<10} Got={r["label"]:<10} Conf={r["confidence"]:.3f}')

## 2. Inference performance benchmark

In [ ]:
import time

test_text = 'I have been feeling overwhelmed and anxious about everything this week.'
n_trials  = 200

times = []
for _ in range(n_trials):
    t0 = time.perf_counter()
    predict_text(test_text, models)
    times.append((time.perf_counter() - t0) * 1000)

times = np.array(times)
print(f'Inference benchmark ({n_trials} trials):')
print(f'  Median : {np.median(times):.2f} ms')
print(f'  Mean   : {np.mean(times):.2f} ms')
print(f'  P95    : {np.percentile(times, 95):.2f} ms')
print(f'  P99    : {np.percentile(times, 99):.2f} ms')
print(f'  Max    : {np.max(times):.2f} ms')
print(f'  Estimated RPS: ~{1000/np.median(times):.0f} requests/sec (single thread)')

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(times, bins=30, color='#7F77DD', alpha=0.85, edgecolor='white')
ax.axvline(np.median(times), color='#1D9E75', linestyle='--', linewidth=1.5, label=f'Median {np.median(times):.2f}ms')
ax.axvline(np.percentile(times,95), color='#D85A30', linestyle='--', linewidth=1.5, label=f'P95 {np.percentile(times,95):.2f}ms')
ax.set_xlabel('Inference time (ms)')
ax.set_ylabel('Count')
ax.set_title('Inference Latency Distribution (200 trials)', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{DOCS}/plot_inference_benchmark.png', bbox_inches='tight')
plt.show()
print(f'\nPlot saved: docs/plot_inference_benchmark.png')

## 3. Run pytest suite

In [ ]:
print('Running pytest...')
result = subprocess.run(
    ['python', '-m', 'pytest', '../tests/test_predict.py', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode == 0:
    print('\n✅ All tests passed!')
else:
    print('\n❌ Some tests failed. Fix before deploying.')
    print(result.stderr[-1000:])

## 4. Flask API integration test (requires server running)

To run this cell:
1. Open a new terminal
2. `cd mental_health_nlp && python app/api.py`
3. Wait for 'Starting Flask dev server on port 5000'
4. Then run this cell

In [ ]:
BASE_URL = 'http://localhost:5000'

try:
    # Health check
    r = requests.get(f'{BASE_URL}/health', timeout=3)
    print(f'GET /health: {r.status_code} {r.json()["status"]}')
    
    # Predict
    payload = {'text': 'I feel hopeless and cannot continue anymore, nothing matters.'}
    r = requests.post(f'{BASE_URL}/predict', json=payload, timeout=5)
    d = r.json()
    print(f'POST /predict: {r.status_code}')
    print(f'  label={d["label"]} conf={d["confidence"]}')
    print(f'  top_tokens={[(t["token"], t["direction"]) for t in d["top_tokens"][:3]]}')
    
    # Batch
    payload = {'texts': ['I feel great today!', 'Feeling hopeless and alone.']}
    r = requests.post(f'{BASE_URL}/predict-batch', json=payload, timeout=5)
    d = r.json()
    print(f'POST /predict-batch: {r.status_code} results={[x["label"] for x in d["results"]]}')
    
    print('\n✅ Flask API integration tests passed!')
except requests.exceptions.ConnectionError:
    print('⚠ Flask server not running. Start it with: python app/api.py')
    print('  Then re-run this cell.')

## 5. Deployment checklist

In [ ]:
import os

checks = [
    ('../models/label_encoder.pkl',    'Label encoder saved'),
    ('../models/tfidf_vectorizer.pkl', 'TF-IDF vectorizer saved'),
    ('../models/lr_tfidf_shap.pkl',    'LR model saved'),
    ('../app/predict.py',              'predict.py exists'),
    ('../app/api.py',                  'api.py exists'),
    ('../app/streamlit_app.py',        'streamlit_app.py exists'),
    ('../app.py',                      'HF entrypoint app.py exists'),
    ('../Dockerfile',                  'Dockerfile exists'),
    ('../hf_requirements.txt',         'HF requirements exist'),
    ('../tests/test_predict.py',       'Test suite exists'),
    ('../docs/MODEL_CARD.md',          'Model Card exists'),
    ('../docs/ethics_note.md',         'Ethics note exists'),
    ('../README.md',                   'README exists'),
]

print('=== PRE-DEPLOYMENT CHECKLIST ===')
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    icon   = '✅' if exists else '❌'
    if not exists: all_ok = False
    print(f'  {icon} {desc}')

print()
if all_ok:
    print('✅ All files in place. Ready to deploy!')
else:
    print('❌ Some files missing. Check the list above.')

## 6. Update README results table

In [ ]:
# Load results table and print as markdown for README
results_path = '../data/processed/results_table.csv'
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print('=== MARKDOWN TABLE FOR README.md ===')
    print()
    print('| Model | F1-macro | CV F1 | Inference | Size |')
    print('|-------|----------|-------|-----------|------|')
    for _, row in df.iterrows():
        f1  = f"{row['f1_macro']:.4f}"      if pd.notna(row.get('f1_macro'))      else 'TBD'
        cv  = f"{row['cv_f1_macro']:.4f}"   if pd.notna(row.get('cv_f1_macro'))   else 'TBD'
        ms  = f"{row['inference_ms']:.1f}ms" if pd.notna(row.get('inference_ms'))  else 'TBD'
        kb  = f"{row['model_size_kb']:.0f}KB" if pd.notna(row.get('model_size_kb')) else 'TBD'
        name = row.get('model') or row.get('model_name','?')
        print(f'| {name} | {f1} | {cv} | {ms} | {kb} |')
else:
    print('[SKIP] results_table.csv not found. Run Week 3 notebook first.')

---
## Week 5 Summary — fill this in

| Task | Status |
|------|--------|
| 45 pytest tests passing | ☐ |
| Flask API routes all return correct codes | ☐ |
| Streamlit app renders locally | ☐ |
| Inference median < 5ms | ☐ |
| HuggingFace Space created | ☐ |
| Live URL tested and working | ☐ |
| README updated with live URL | ☐ |
| GitHub README demo GIF added | ☐ |

**Your interview one-liner:**
> 'The demo is live at [HF_URL]. I can open it right now if you'd like to see it running.'
